# LFCC + temporal Swin — full training from scratch

Trains the frozen `ecgdata_lfcc_small_swin_v1` protocol end to end and reports its results:

1. `ECGData.mat` → the exact 1,620 baseline windows (hash-checked against the frozen baseline)
2. LFCC features (13 frames × 1 lead × 16 cepstra per window)
3. 15 Swin fits: seeds 0/1/2 × five patient-separated folds, 40 epochs each, fixed settings
4. Window and patient-vote accuracy / macro-F1 per seed and mean, per-class recall, confusion
   matrices, training curves, patient bootstrap intervals

Only the Swin arm is trained (no logistic control). Outputs go to a **new** directory,
`attention method/results/ecgdata_lfcc_swin_retrain/`; the completed v1 run is never touched.
Each fit is checkpointed every epoch, so re-running the training cell resumes rather than restarts.

**Upload the whole `reproduce_swin/` folder** (zip it as `reproduce_swin.zip`). Keep the layout;
the code checks file hashes, so do not edit the files. Nothing else from the repository is needed.

```
reproduce_swin/
├── lfcc_swin_training.ipynb                 this notebook
├── ECGData.mat                              raw ECGs (162 lead rows, 128 Hz)
├── ecgvmd/                                  ECGData loader, patient map, window segmentation
├── results/patient_vmd_wst_vqc/             frozen windows/patient folds to check against
│   └── manifest.json, folds.csv, features.npz
└── attention method/
    ├── ecgdata_protocol.json                frozen LFCC, model and training settings
    └── attention_ecg/                       LFCC extractor, Swin model, training, metrics
```

**Exactness:** the windows and LFCC features are deterministic and checked against the frozen hashes.
The trained weights are deterministic *on a given device and library stack*, but a different GPU or
PyTorch version (e.g. a Colab T4 vs the original RTX 5070 / PyTorch 2.7.1+cu128) will give slightly
different weights, so expect results close to — not bit-identical with — the original 83.35% / 0.8006 / 94.17%.

## 0. Colab only: unpack the files and pin numerical libraries

In [ ]:
# Uncomment in Colab, run once, then Runtime -> Restart session, then run the rest.
# Upload reproduce_swin.zip to /content first (Files panel, or copy it from Google Drive).
# !unzip -q -o /content/reproduce_swin.zip -d /content
# !pip install -q numpy==2.5.2 scipy==1.18.1 scikit-learn==1.9.0 pandas==3.0.5

## 1. Settings and environment

In [ ]:
import os, sys, json, time, platform
from pathlib import Path

# Required by deterministic CUDA matrix multiplies; must be set before torch initializes CUDA.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

here = Path.cwd().resolve()
candidates = (here / 'attention method', here, here / 'reproduce_swin/attention method',
              Path('/content/reproduce_swin/attention method'), Path('/content/attention method'))
ROOT = next(p for p in candidates if (p / 'attention_ecg').is_dir())
REPO = ROOT.parent
local_torch = ROOT / '.venv/torch-cu128'          # local machine only; Colab has torch built in
for p in ([str(local_torch)] if local_torch.is_dir() else []) + [str(ROOT), str(REPO)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np, scipy, sklearn, pandas as pd
import matplotlib.pyplot as plt
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ROOT / 'results/ecgdata_lfcc_swin_retrain'   # new directory; never the completed v1 run
SEEDS = [0, 1, 2]
FOLDS = [0, 1, 2, 3, 4]

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

assert OUT.resolve() != (ROOT / 'results/ecgdata_lfcc_swin_v1').resolve()
env = dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__,
           sklearn=sklearn.__version__, torch=torch.__version__, device=DEVICE,
           device_name=torch.cuda.get_device_name() if DEVICE == 'cuda' else platform.processor())
print('attention method:', ROOT)
print(env)

## 2. Feature extraction from scratch: windows and LFCC

`prepare_cache` regenerates the 1,620 windows from `ECGData.mat`, requires them to match the frozen
baseline hashes and patient folds exactly, extracts LFCC per lead row, and saves
`features/ecgdata_lfcc_v1/`. If that cache already exists, it is verified and reused.

In [ ]:
from attention_ecg.ecgdata import PROTOCOL, read_json, prepare_cache, verify_cache, atomic_json

protocol = read_json(PROTOCOL)
CLASSES = protocol['classes']
t0 = time.perf_counter()
print(prepare_cache(REPO / 'ECGData.mat'))
cache_manifest, data = verify_cache()
print(f'{time.perf_counter() - t0:.1f} s | X {data["X"].shape} | patients {len(np.unique(data["patients"]))}')
print('cepstral:', protocol['cepstral'])
print('model:   ', protocol['model'])
print('training:', protocol['training'])

In [ ]:
split = pd.DataFrame(dict(fold=data['fold'], cls=[CLASSES[k] for k in data['y']], patient=data['patients']))
table = split.pivot_table(index='fold', columns='cls', values='patient', aggfunc='count')
table['patients'] = split.groupby('fold').patient.nunique()
table

What the extraction did to each window, shown for one window per class: a 3.9 s z-scored ECG window
is cut into 13 one-second Hann frames (hop 0.25 s); each frame's power spectrum is pooled into 32 linear
bands from 0–64 Hz, logged, and DCT-transformed; c0–c15 are kept. The cached features are recomputed here
from the raw windows and must match exactly.

In [ ]:
from attention_ecg.ecgdata import matched_windows
from attention_ecg.cepstral import CepstralConfig, cepstral_features

windows, _ = matched_windows(REPO / 'ECGData.mat', protocol)
cfg = CepstralConfig(**protocol['cepstral'])
fig, axes = plt.subplots(2, 3, figsize=(15, 5))
for k in range(3):
    i = np.flatnonzero(data['y'] == k)[0]
    lfcc, meta = cepstral_features(windows[i][:, None], cfg)
    assert np.array_equal(lfcc, data['X'][i])
    axes[0, k].plot(np.arange(len(windows[i])) / cfg.fs, windows[i], lw=.7)
    axes[0, k].set(title=f'{CLASSES[k]}: patient {data["patients"][i]}', xlabel='seconds')
    im = axes[1, k].imshow(lfcc[:, 0].T, aspect='auto', origin='lower', cmap='coolwarm')
    axes[1, k].set(xlabel='frame (0.25 s hop)', ylabel='cepstral coefficient')
    fig.colorbar(im, ax=axes[1, k])
fig.tight_layout()
print('LFCC per window:', data['X'].shape[1:], '(frames, leads, coefficients)')

## 3. Train the 15 Swin fits

Each fit (`fit_swin`): fold-training-only normalization; seeded initialization; AdamW (lr 1e-3, weight
decay 0.01); cosine decay to 1e-4 over 40 epochs; batch 32; class-weighted cross-entropy from the
training partition; gradient clipping at norm 1; no early stopping. The held-out fold is predicted
once, after epoch 40. A completed fit is verified by checksum and skipped on re-run.

In [ ]:
from attention_ecg.ecgdata_training import fit_swin, verify_trial

OUT.mkdir(parents=True, exist_ok=True)
atomic_json(OUT / 'environment.json', dict(env, protocol=protocol, seeds=SEEDS, folds=FOLDS))
log = []
t_all = time.perf_counter()
for fold in FOLDS:
    for seed in SEEDS:
        folder = OUT / f'swin_seed{seed}_fold{fold}'
        done = verify_trial(folder)
        t0 = time.perf_counter()
        fit_swin(folder, data, fold, seed, protocol, DEVICE)
        fit = read_json(folder / 'fit.json')
        log.append(dict(seed=seed, fold=fold, reused=done, fit_seconds=fit['seconds']))
        print(f'seed {seed} fold {fold}: {"reused" if done else "trained"} '
              f'({time.perf_counter() - t0:.1f} s wall, final loss {read_json(folder / "history.json")[-1]["loss"]:.4f})')
print(f'total wall time {time.perf_counter() - t_all:.1f} s')
pd.DataFrame(log)

## 4. Results

Window metrics pool all 1,620 out-of-fold windows. Patient vote takes each patient's majority predicted
class (ties → lowest class index). The mean averages per-seed results; it is not a seed ensemble.

In [ ]:
from attention_ecg.ecgdata_report import scores, patient_confusions, bootstrap_comparison
from attention_ecg.ecgdata_training import verify_trial

preds, probs = {}, {}
for seed in SEEDS:
    pred = np.full(len(data['y']), -1)
    prob = np.full((len(data['y']), 3), np.nan)
    for fold in FOLDS:
        folder = OUT / f'swin_seed{seed}_fold{fold}'
        assert verify_trial(folder)
        with np.load(folder / 'predictions.npz') as z:
            assert np.array_equal(z['indices'], np.flatnonzero(data['fold'] == fold))
            pred[z['indices']], prob[z['indices']] = z['pred'], z['probabilities']
    preds[seed], probs[seed] = pred, prob
covered = all((p >= 0).all() for p in preds.values())
assert covered, 'Results need all five folds for every seed'

rows, per_seed = [], {}
for seed, pred in preds.items():
    w, p = patient_confusions(data['y'], pred, data['patients'])
    per_seed[seed] = dict(window=scores(w.sum(0)), patient_vote=scores(p.sum(0)))
    r = per_seed[seed]
    rows.append(dict(seed=seed, window_accuracy=r['window']['accuracy'], window_macro_f1=r['window']['macro_f1'],
                     patient_vote_accuracy=r['patient_vote']['accuracy'], patient_vote_macro_f1=r['patient_vote']['macro_f1'],
                     **{f'{c}_window_recall': r['window']['sensitivity'][k] for k, c in enumerate(CLASSES)}))
results = pd.DataFrame(rows).set_index('seed')
results.loc['mean'] = results.mean()
results.loc['sd (seeds)'] = results.loc[SEEDS].std(ddof=1)

# Original v1 run (RTX 5070, PyTorch 2.7.1+cu128), from reports/ecgdata_lfcc_swin_v1/results.md.
original = pd.Series(dict(window_accuracy=0.8335, window_macro_f1=0.8006,
                          patient_vote_accuracy=0.9417, patient_vote_macro_f1=0.9269), name='original v1 mean')
display(results)
display(pd.DataFrame([results.loc['mean', original.index], original]))

In [ ]:
e = protocol['evaluation']
boot = bootstrap_comparison(data['y'], data['patients'], {'swin': [preds[s] for s in SEEDS]},
                            e['bootstrap_resamples'], e['bootstrap_seed'])
pd.DataFrame([dict(metric=m, mean=results.loc['mean', m], low=b[0], high=b[1])
              for m, b in boot['intervals_95']['swin'].items()]).set_index('metric')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cm = np.sum([per_seed[s]['window']['confusion_matrix'] for s in SEEDS], axis=0)
cm = cm / cm.sum(1, keepdims=True)
axes[0].imshow(cm, vmin=0, vmax=1, cmap='Blues')
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center', color='white' if cm[i, j] > .5 else 'black')
axes[0].set(xticks=range(3), yticks=range(3), xticklabels=CLASSES, yticklabels=CLASSES,
            xlabel='predicted', ylabel='true', title='Window confusion (row-normalized, seeds pooled)')

hist = pd.DataFrame([dict(seed=s, fold=f, **h) for s in SEEDS for f in FOLDS
                     for h in read_json(OUT / f'swin_seed{s}_fold{f}' / 'history.json')])
for s, g in hist.groupby('seed'):
    axes[1].plot(g.groupby('epoch').loss.mean(), label=f'seed {s}')
axes[1].set(yscale='log', xlabel='epoch', ylabel='class-weighted training CE', title='Mean training loss across folds')
axes[1].legend()
axes[2].plot(hist[(hist.seed == SEEDS[0]) & (hist.fold == FOLDS[0])].set_index('epoch').lr)
axes[2].set(xlabel='epoch', ylabel='learning rate', title='Cosine schedule')
fig.tight_layout()

Limitations (see `reports/ecgdata_lfcc_swin_v1/results.md`): 80 patients; diagnosis is confounded with
source database; exploratory, not external validation. The training curves are training loss only and
do not identify an optimal epoch. Bootstrap intervals are conditional on these predictions and exclude
retraining uncertainty. ARR/CHF/NSR classification, not MI/OMI.